In [1]:
# Install psycopg2 (PostgreSQL connector for Python)
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "psycopg2-binary"])
print("✓ psycopg2 installed successfully")

✓ psycopg2 installed successfully


In [2]:
import pandas as pd
import psycopg2
from psycopg2 import sql

# Read cleaned CSV data
df = pd.read_csv('../data/processed/master_delivery_data.csv')

print(f"Loaded CSV with {df.shape[0]:,} rows and {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")

# ============================================================
# DATABASE CONNECTION DETAILS
# ============================================================
# CHANGE THESE TO YOUR POSTGRESQL CREDENTIALS
# ============================================================

DB_CONFIG = {
    'host': 'localhost',      # Usually localhost
    'port': 5432,             # Default PostgreSQL port
    'database': 'logitrack_db',  # Your database name
    'user': 'postgres',       # Your PostgreSQL username
    'password': 'killua'  
}

print("\nDatabase Configuration:")
print(f"Host: {DB_CONFIG['host']}")
print(f"Database: {DB_CONFIG['database']}")
print(f"User: {DB_CONFIG['user']}")

Loaded CSV with 180,519 rows and 60 columns
Columns: ['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Produ

In [3]:
try:
    # Connect to PostgreSQL
    connection = psycopg2.connect(**DB_CONFIG)
    cursor = connection.cursor()
    
    print("✓ Connected to PostgreSQL successfully")
    
    # ============================================================
    # INSERT DATA
    # ============================================================
    
    print("\nInserting data into 'orders' table...")
    
    # Prepare column names
    columns = df.columns.tolist()
    columns_str = ', '.join(columns)
    
    # Insert data row by row
    insert_query = f"""
        INSERT INTO orders ({columns_str})
        VALUES ({', '.join(['%s'] * len(columns))})
    """
    
    for idx, row in df.iterrows():
        cursor.execute(insert_query, tuple(row))
        
        # Print progress every 10,000 rows
        if (idx + 1) % 10000 == 0:
            print(f"  Inserted {idx + 1:,} rows...")
    
    # Commit the transaction
    connection.commit()
    print(f"\n✓ Successfully inserted {df.shape[0]:,} rows into orders table")
    
except (Exception, psycopg2.Error) as error:
    print(f"✗ Error while connecting to PostgreSQL or inserting data:")
    print(error)
    connection.rollback()
    
finally:
    if connection:
        cursor.close()
        connection.close()
        print("\n✓ PostgreSQL connection closed")

✓ Connected to PostgreSQL successfully

Inserting data into 'orders' table...
✗ Error while connecting to PostgreSQL or inserting data:
syntax error at or near "for"
LINE 2:         INSERT INTO orders (Type, Days for shipping (real), ...
                                               ^


✓ PostgreSQL connection closed


In [4]:
try:
    # Reconnect to verify
    connection = psycopg2.connect(**DB_CONFIG)
    cursor = connection.cursor()
    
    print("Verifying data in database...\n")
    
    # Count total rows
    cursor.execute("SELECT COUNT(*) FROM orders;")
    total_rows = cursor.fetchone()[0]
    print(f"✓ Total rows in orders table: {total_rows:,}")
    
    # Sample data
    print("\n" + "="*70)
    print("SAMPLE DATA (First 5 rows):")
    print("="*70)
    cursor.execute("""
        SELECT order_id, shipping_mode, delivery_status, late_delivery_risk, 
               customer_state, category_name, sales_per_customer
        FROM orders
        LIMIT 5;
    """)
    
    for row in cursor.fetchall():
        print(row)
    
    # Data by Shipping Mode
    print("\n" + "="*70)
    print("ORDERS BY SHIPPING MODE:")
    print("="*70)
    cursor.execute("""
        SELECT shipping_mode, COUNT(*) as count
        FROM orders
        GROUP BY shipping_mode;
    """)
    
    for row in cursor.fetchall():
        print(f"{row[0]}: {row[1]:,}")
    
    # Late Delivery Summary
    print("\n" + "="*70)
    print("LATE DELIVERY SUMMARY:")
    print("="*70)
    cursor.execute("""
        SELECT 
            SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) as late_orders,
            SUM(CASE WHEN late_delivery_risk = 0 THEN 1 ELSE 0 END) as on_time_orders,
            ROUND(100.0 * SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as late_rate_percent
        FROM orders;
    """)
    
    row = cursor.fetchone()
    print(f"Late Deliveries: {row[0]:,}")
    print(f"On-Time Deliveries: {row[1]:,}")
    print(f"Late Rate: {row[2]}%")
    
    connection.commit()
    
except (Exception, psycopg2.Error) as error:
    print(f"Error: {error}")
    
finally:
    if connection:
        cursor.close()
        connection.close()
        print("\n✓ Verification complete")

Verifying data in database...

✓ Total rows in orders table: 0

SAMPLE DATA (First 5 rows):

ORDERS BY SHIPPING MODE:

LATE DELIVERY SUMMARY:
Error: unsupported format string passed to NoneType.__format__

✓ Verification complete
